# 1 — Stage 2 setup and preflight
This reuses the Stage 1 ID environment and freezes one idle physical A100. It makes no benchmark measurements.


In [ ]:
import os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; OUT=Path.home()/"stage2"
for path in (R, PY):
    if not path.exists(): raise SystemExit(f"STOP: missing {path}. On a fresh machine, complete Stage 1 notebook 01 first.")
OUT.mkdir(exist_ok=True)
print("Using anonymous source")
print("repository SHA","anonymous-source")


In [ ]:
# Choose the lowest-memory A100 and freeze its physical ID for all later notebooks.
raw=subprocess.run(["nvidia-smi","--query-gpu=index,name,memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip().splitlines()
rows=[]
for line in raw:
    idx,name,mem,util=[x.strip() for x in line.split(",",3)]; rows.append((int(mem),int(util),idx,name))
print(*raw,sep="\n"); chosen=min(rows); GPU=chosen[2]
if chosen[0] > 2000 or chosen[1] > 5: raise SystemExit(f"STOP: best GPU is not idle: {chosen}")
(Path.home()/"stage2_gpu.txt").write_text(GPU); print("PASS selected physical GPU",GPU)


In [ ]:
# Run tests and authentication in the exact headless subprocess environment used later.
test_env=os.environ.copy(); test_env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(PY),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=test_env,check=True)
code="from huggingface_hub import HfApi; print(HfApi().whoami().get(\"name\"))"
subprocess.run([str(PY),"-c",code],env=test_env,check=True)
print("PASS: preflight complete. Continue to notebook 02.")
